# Local AgentForge SWE-bench Smoke

This notebook is for trying the repository on a laptop before moving to HPC.

It starts with an offline smoke test that does not call a model, Hugging Face, Docker, or the official harness. Optional cells then show how to install and run Kwai/Klear's official `mini-swe-agent-plus` harness.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
assert (ROOT / 'pyproject.toml').exists(), ROOT

env = os.environ.copy()
env['PYTHONPATH'] = str(ROOT / 'src') + (':' + env['PYTHONPATH'] if env.get('PYTHONPATH') else '')
env.setdefault('UV_CACHE_DIR', str(ROOT / '.uv-cache'))
print(ROOT)

## 1. Offline Smoke Test

This creates two tiny SWE-bench-shaped records locally and runs the mock collector. It verifies task loading, trajectory writing, prediction JSONL writing, and summaries.

In [ ]:
smoke_dir = ROOT / 'data' / 'processed' / 'notebook_smoke'
smoke_dir.mkdir(parents=True, exist_ok=True)
dataset_path = smoke_dir / 'local_tasks.jsonl'

rows = [
    {
        'instance_id': 'local__demo-1',
        'repo': 'local/demo',
        'base_commit': '0' * 40,
        'problem_statement': 'Synthetic local smoke task.',
        'patch': 'diff --git a/demo.py b/demo.py\n--- a/demo.py\n+++ b/demo.py\n',
    },
    {
        'instance_id': 'local__demo-2',
        'repo': 'local/demo',
        'base_commit': '1' * 40,
        'problem_statement': 'Second synthetic local smoke task.',
        'patch': 'diff --git a/demo2.py b/demo2.py\n--- a/demo2.py\n+++ b/demo2.py\n',
    },
]
dataset_path.write_text('\n'.join(json.dumps(row) for row in rows) + '\n')

cmd = [
    sys.executable,
    '-m',
    'debug_depo.rollout',
    '--dataset',
    str(dataset_path),
    '--output-dir',
    str(smoke_dir / 'rollouts'),
    '--mock',
    '--mock-patch',
    'gold',
    '--limit',
    '2',
    '--overwrite',
    '--no-progress',
]
subprocess.run(cmd, cwd=ROOT, env=env, check=True)

In [ ]:
summary_path = smoke_dir / 'rollouts' / 'summary.json'
predictions_path = smoke_dir / 'rollouts' / 'predictions.jsonl'

summary = json.loads(summary_path.read_text())
predictions = [json.loads(line) for line in predictions_path.read_text().splitlines()]

print(json.dumps({k: summary[k] for k in ['n_tasks', 'n_completed', 'n_errors', 'n_with_patch', 'predictions_path']}, indent=2))
predictions

## 2. Optional: Install Official mini-swe-agent-plus

Kwai/Klear's official runnable SWE harness is `mini-swe-agent-plus`. Keep this disabled until you want to clone/install it into the repo `.venv`.

In [ ]:
RUN_INSTALL = True

if RUN_INSTALL:
    subprocess.run(['bash', 'scripts/install_mini_swe_agent_plus.sh'], cwd=ROOT, env=env, check=True)
else:
    print('Skipped. Set RUN_INSTALL = True to clone and install the official harness.')

## 3. Optional: Run One Real Harness Task

This requires:

- `mini-swe-agent-plus` installed from the previous cell
- Docker running
- an OpenAI-compatible model server for the AgentForge SFT model
- enough disk space for SWE-bench images

On a Mac, treat this as a small functional test only. The full benchmark belongs on an x86_64 Linux/HPC setup.

In [ ]:
RUN_REAL_MINI = True
STRICT_REAL_MINI = os.environ.get('STRICT_REAL_MINI', '0') == '1'


def show_real_mini_summary(output_dir):
    summary_path = output_dir / 'summary.json'
    predictions_path = output_dir / 'predictions.jsonl'
    summary = json.loads(summary_path.read_text())

    compact = {
        key: summary[key]
        for key in ['n_tasks', 'n_completed', 'n_errors', 'n_with_patch', 'predictions_path']
    }
    compact['results'] = summary['results']
    print(json.dumps(compact, indent=2))

    if summary['n_errors'] or summary['n_with_patch'] != summary['n_tasks']:
        print('\nMini-swe finished, but at least one instance did not submit a patch.')
        print('This matches the official batch behavior: record the exit status, keep predictions.jsonl, and inspect artifacts.')
        print(f'Summary: {summary_path}')
        print(f'Predictions: {predictions_path}')
        for result in summary['results']:
            if result.get('status') == 'error' or not result.get('patch_source'):
                print('\nInstance:', result.get('instance_id'))
                print('Status:', result.get('status'))
                if result.get('mini_swe_exit_status'):
                    print('mini-swe exit status:', result['mini_swe_exit_status'])
                    print('exit status file:', result.get('mini_swe_exit_status_path'))
                print('trajectory:', result.get('trajectory_path'))

        if STRICT_REAL_MINI:
            raise RuntimeError(f'Real mini-swe smoke did not produce patches for every task. Inspect {summary_path}')
    return summary


if RUN_REAL_MINI:
    local_model = os.environ.get('LOCAL_LLM_MODEL', 'mlx-community/Qwen2.5-Coder-7B-Instruct-4bit')
    output_dir = ROOT / 'data' / 'processed' / 'notebook_real_miniswe'
    real_env = env.copy()
    real_env.update({
        'HARNESS': 'mini-swe-agent-plus',
        'AGENTFORGE_MODEL': local_model,
        'MINI_SWE_MODEL': f'hosted_vllm/{local_model}',
        'LLM_BASE_URL': 'http://127.0.0.1:8000/v1',
        'LLM_API_KEY': 'local',
        'LIMIT': '1',
        'MAX_STEPS': '10',
        'OUTPUT_DIR': str(output_dir),
        'OVERWRITE': '1',
        'STREAM_OUTPUT': '1',
    })
    subprocess.run(['bash', 'scripts/collect_rollouts.sh'], cwd=ROOT, env=real_env, check=True)
    real_summary = show_real_mini_summary(output_dir)
else:
    print('Skipped. Set RUN_REAL_MINI = True after starting your model server and Docker.')


## 4. Optional: Official SWE-bench Evaluation

This consumes `predictions.jsonl` and runs the official SWE-bench Docker evaluator. For a first local test, evaluate only the tiny output from `RUN_REAL_MINI` or a gold-patch smoke output. The full 500-instance evaluation is for HPC.

In [ ]:
RUN_EVAL = True

if RUN_EVAL:
    eval_env = env.copy()
    eval_env.update({
        'PREDICTIONS_PATH': str(ROOT / 'data' / 'processed' / 'notebook_real_miniswe' / 'predictions.jsonl'),
        'MAX_WORKERS': '1',
        'RUN_ID': 'notebook_one_task',
    })
    subprocess.run(['bash', 'scripts/evaluate_all.sh'], cwd=ROOT, env=eval_env, check=True)
else:
    print('Skipped. Set RUN_EVAL = True only when Docker and predictions are ready.')